# Loading Data

In [159]:
import pandas as pd
import numpy as np
import ast

In [160]:
df = pd.read_csv(r"D:\all_league\all_league_all_matches.csv", low_memory=False)

In [161]:
people = pd.read_csv(r'D:\all_league\people3.csv')

# Cleaning data

renaming columns

In [162]:
df =  df.rename(columns={'striker':'batter'})

Dropping unnecessary columns

In [163]:
df.drop(columns = ['other_wicket_type', 'other_player_dismissed', 'penalty'], inplace=True)

Fill null values

In [164]:
df['wides'] = df['wides'].fillna(0)
df['noballs'] = df['noballs'].fillna(0)
df['byes'] = df['byes'].fillna(0)
df['legbyes'] = df['legbyes'].fillna(0)

Create new columns

In [165]:
df['batter_balls_faced'] = np.where(df['wides']==0, 1, 0)
df['bowler_runs'] = df['runs_off_bat'] + df['wides'] + df['noballs']

Resolving player name for consistency

In [166]:
# Create a dictionary mapping unique_name to key_cricinfo
name_to_id = dict(zip(people['unique_name'], people['key_cricinfo']))
# Map batter and bowler names to their respective IDs using the dictionary
df['batter_id'] = df['batter'].map(name_to_id)
df['bowler_id'] = df['bowler'].map(name_to_id)
df['non_striker_id'] = df['non_striker'].map(name_to_id)

# Batting Scorecard

In [167]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'batter', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'wicket_type', 'player_dismissed', 'league', 'batter_balls_faced',
       'bowler_runs', 'batter_id', 'bowler_id', 'non_striker_id'],
      dtype='object')

In [168]:
def process_batting_scorecard(df):
    if df.empty:
        return pd.DataFrame()

    batting_stats = {}

    for _, ball in df.iterrows():
        match_id = ball['match_id']
        match_date = ball['start_date']
        venue = ball['venue']

        # For the striker, use the batter_id column.
        striker_id = ball['batter_id']
        striker_name = ball['batter']
        # For the non-striker, now use both the name and the id.
        non_striker_name = ball.get('non_striker', None)
        non_striker_id = ball.get('non_striker_id', None)

        bowler_name = ball['bowler']
        batting_team = ball['batting_team']
        bowling_team = ball['bowling_team']
        league_name = ball['league']

        # Process the striker's record using striker_id as key
        key_striker = (match_id, batting_team, striker_id)
        if key_striker not in batting_stats:
            batting_stats[key_striker] = {
                'batsman_name': striker_name,
                'bowler_name': "",
                'runs': 0,
                'balls': 0,
                'fours': 0,
                'sixes': 0,
                'isOut': False,
                'wicketType': 'not out',
                'innings': ball['innings'],
                'p_bat_id': striker_id,
                'p_bowl_id': None,
                'bat_team': batting_team,
                'bowl_team': bowling_team,
                'date': match_date,
                'venue': venue,
                'league': league_name
            }
        stats = batting_stats[key_striker]

        # Update runs and boundaries for striker
        runs = ball['runs_off_bat']
        stats['runs'] += runs
        if runs == 4:
            stats['fours'] += 1
        elif runs == 6:
            stats['sixes'] += 1

        # Count balls faced (excluding wides and no-balls)
        if ball['batter_balls_faced'] == 1:
            stats['balls'] += 1

        # Check for dismissal of the striker
        if pd.notna(ball['wicket_type']):
            if ball['player_dismissed'] == striker_name:
                stats['isOut'] = True
                stats['wicketType'] = ball['wicket_type']
                # Only assign bowler if it's not a run out
                if ball['wicket_type'] != 'run out':
                    stats['bowler_name'] = bowler_name
                    stats['p_bowl_id'] = ball['bowler_id']
                else:
                    stats['bowler_name'] = ""
                    stats['p_bowl_id'] = None

        batting_stats[key_striker] = stats

        # Now, separately, check if the non-striker got run out.
        if pd.notna(ball['wicket_type']) and ball['wicket_type'] == 'run out' and (non_striker_name is not None and ball['player_dismissed'] == non_striker_name):
            # Use non_striker_id for the key so that we do not set batter_id as non_striker name
            key_non_striker = (match_id, batting_team, non_striker_id)
            if key_non_striker in batting_stats:
                # Update the existing record
                batting_stats[key_non_striker]['isOut'] = True
                batting_stats[key_non_striker]['wicketType'] = 'run out'
                batting_stats[key_non_striker]['bowler_name'] = ""
                batting_stats[key_non_striker]['p_bowl_id'] = None
            else:
                # Create new record for non-striker using non_striker_id
                batting_stats[key_non_striker] = {
                    'batsman_name': non_striker_name,
                    'bowler_name': "",
                    'runs': 0,
                    'balls': 0,
                    'fours': 0,
                    'sixes': 0,
                    'isOut': True,  # Mark as out because of run out
                    'wicketType': 'run out',
                    'innings': ball['innings'],
                    'p_bat_id': non_striker_id,  # Use the non_striker id rather than name
                    'p_bowl_id': None,
                    'bat_team': batting_team,
                    'bowl_team': bowling_team,
                    'date': match_date,
                    'venue': venue,
                    'league': league_name
                }

    # Convert the batting_stats dictionary into a DataFrame.
    results = []
    for (match_id, team, player), stats in batting_stats.items():
        strike_rate = (stats['runs'] / stats['balls'] * 100) if stats['balls'] > 0 else 0
        results.append({
            'match_id': match_id,
            'date': stats['date'],
            'venue': stats['venue'],
            'inns': stats['innings'],
            'batter_id': player,
            'batter': stats['batsman_name'],
            'bowler': stats['bowler_name'],
            'bowler_id': stats['p_bowl_id'],
            'runs': stats['runs'],
            'balls': stats['balls'],
            'fours': stats['fours'],
            'sixes': stats['sixes'],
            'sr': round(strike_rate, 2),
            'isOut': stats['isOut'],
            'wicketType': stats['wicketType'],
            'bat_team': stats['bat_team'],
            'bowl_team': stats['bowl_team'],
            'league': stats['league'],
        })

    return pd.DataFrame(results)

# Usage example:
batting_card = process_batting_scorecard(df)
batting_card.sort_values(by=['date', 'match_id', 'inns'], inplace=True)
batting_card['bowler'] = batting_card['bowler'].replace("", pd.NA)

In [176]:
len(batting_card.match_id.unique()), len(df.match_id.unique())

(7323, 7323)

In [177]:
# Only convert non-null values to int
batting_card['batter_id'] = batting_card['batter_id'].apply(lambda x: int(x) if pd.notna(x) else x)

In [178]:
batting_card

,match_id,date,venue,inns,batter_id,batter,bowler,bowler_id,runs,balls,fours,sixes,sr,isOut,wicketType,bat_team,bowl_team,league
92175,211048,2005-02-17,Eden Park,1,5390,AC Gilchrist,KD Mills,37740.0,1,3,0,0,33.33,True,caught,Australia,New Zealand,T20IS
92176,211048,2005-02-17,Eden Park,1,4578,MJ Clarke,DR Tuffey,38620.0,7,4,0,1,175.00,True,caught,Australia,New Zealand,T20IS
92177,211048,2005-02-17,Eden Park,1,7702,A Symonds,KD Mills,37740.0,32,13,4,2,246.15,True,caught,Australia,New Zealand,T20IS
92178,211048,2005-02-17,Eden Park,1,7133,RT Ponting,<NA>,NaN,98,55,8,5,178.18,False,not out,Australia,New Zealand,T20IS
92179,211048,2005-02-17,Eden Park,1,6513,DR Martyn,KD Mills,37740.0,3,5,0,0,60.00,True,bowled,Australia,New Zealand,T20IS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47458,1473458,2025-04-07,Wankhede Stadium,2,1287032,Naman Dhir,KH Pandya,471342.0,11,6,1,1,183.33,True,caught,Mumbai Indians,Royal Challengers Bangalore,IPL
47459,1473458,2025-04-07,Wankhede Stadium,2,502714,MJ Santner,KH Pandya,471342.0,8,4,0,1,200.00,True,caught,Mumbai Indians,Royal Challengers Bangalore,IPL
47460,1473458,2025-04-07,Wankhede Stadium,2,447261,DL Chahar,KH Pandya,471342.0,0,1,0,0,0.00,True,caught,Mumbai Indians,Royal Challengers Bangalore,IPL
47461,1473458,2025-04-07,Wankhede Stadium,2,277912,TA Boult,<NA>,NaN,1,1,0,0,100.00,False,not out,Mumbai Indians,Royal Challengers Bangalore,IPL


In [183]:
# Get unique batters from both dataframes
batting_card_batters = set(batting_card['batter'].unique())
df_batters = set(df['batter'].unique())

# Find batters that are in batting_card but not in df
extra_batters = batting_card_batters - df_batters

# Display the extra batters
print(f"Extra batters in batting_card: {extra_batters} \n")
print(f"Count: {len(extra_batters)}")

Extra batters in batting_card: {'Shafiqul Islam', 'JM Herrick', 'Ali Shafiq', 'M Tharupathi', 'Adeel Malik', 'Rohan Suresh'} 

Count: 6


# Bowling Card

In [184]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'batter', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'wicket_type', 'player_dismissed', 'league', 'batter_balls_faced',
       'bowler_runs', 'batter_id', 'bowler_id', 'non_striker_id'],
      dtype='object')

In [185]:
df['bowlruns'] = df['runs_off_bat'] + df['wides'] + df['noballs']

In [186]:
df['wicket_type'].unique()

array([nan, 'caught', 'bowled', 'lbw', 'run out', 'stumped', 'hit wicket',
       'caught and bowled', 'retired hurt', 'obstructing the field',
       'retired out', 'retired not out'], dtype=object)

In [194]:
import pandas as pd
import numpy as np

def calculate_bowling_card(df):
    # Ensure necessary columns are of the right type
    df = df.copy()

    # Convert columns to appropriate types
    df['noballs'] = df['noballs'].fillna(0).astype(float)
    df['wides'] = df['wides'].fillna(0).astype(float)
    df['runs_off_bat'] = df['runs_off_bat'].fillna(0).astype(float)
    df['bowler_runs'] = df['bowler_runs'].fillna(0).astype(float)

    # Ensure ball column is string
    df['ball'] = df['ball'].astype(str)

    # Create legal delivery indicator
    df['legal_delivery'] = (df['noballs'] == 0) & (df['wides'] == 0)

    # Create valid wicket indicator (excluding run outs)
    df['valid_wicket'] = (df['wicket_type'].notna()) & (df['wicket_type'] != '') & (df['wicket_type'] != 'run out')

    # Group by match, bowler, bowler ID, innings
    bowling_card = df.groupby(['match_id', 'bowler', 'bowler_id', 'bowling_team', 'batting_team', 'innings'])

    # Compute required statistics
    summary = bowling_card.agg(
        overs=('legal_delivery', lambda x: f"{x.sum() // 6}.{x.sum() % 6}"),  # Overs in x.y format
        balls=('ball', 'count'),  # Total deliveries (including wides and no-balls)
        wickets=('valid_wicket', 'sum'),  # Count of valid wickets (excluding run outs)
        runs=('bowler_runs', 'sum'),  # Total runs conceded
        lbw_or_bowled=('wicket_type', lambda x: ((x == 'lbw') | (x == 'bowled')).sum()),
        dot_balls=('legal_delivery', lambda x: ((x == True) & (df.loc[x.index, 'runs_off_bat'] == 0)).sum())  # Legal deliveries with no runs off bat
    ).reset_index()

    # Calculate economy rate
    def calculate_economy(row):
        # Handle cases with 0 overs
        if row['overs'] == '0.0':
            return 0.0  # Or np.inf, depending on how you want to handle this

        # Calculate total overs as a float
        overs_parts = row['overs'].split('.')
        total_overs = int(overs_parts[0]) + int(overs_parts[1]) / 6

        # Prevent division by zero
        return row['runs'] / total_overs if total_overs > 0 else 0.0

    summary['economy'] = summary.apply(calculate_economy, axis=1)

    # Calculate maidens
    maiden_overs = df.groupby(['match_id', 'bowler', 'bowler_id', 'innings', df['ball'].str.split('.').str[0].astype(int)]).agg(
        over_runs=('runs_off_bat', 'sum')
    ).reset_index()

    maidens = maiden_overs.groupby(['match_id', 'bowler', 'bowler_id', 'innings'])['over_runs'].apply(lambda x: (x == 0).sum()).reset_index()
    summary = summary.merge(maidens, on=['match_id', 'bowler', 'bowler_id', 'innings'], how='left')
    summary.rename(columns={'over_runs': 'maidens'}, inplace=True)

    # Add venue and date and league
    match_info = df.groupby('match_id').agg(
        venue=('venue', 'first'),
        date=('start_date', 'first'),
        league=('league', 'first'),
    ).reset_index()
    summary = summary.merge(match_info, on='match_id', how='left')

    # Determine the first over bowled by each player in each innings
    first_over = df[df['legal_delivery']].groupby(['match_id', 'bowler', 'bowler_id', 'innings'])['ball'].apply(
        lambda x: min(int(str(val).split('.')[0]) for val in x)
    ).reset_index()

    summary = summary.merge(first_over, on=['match_id', 'bowler', 'bowler_id', 'innings'], how='left')

    # Sort by match, innings, and first over
    summary = summary.sort_values(by=['match_id', 'innings', 'ball'])

    summary.drop(columns=['ball'], inplace=True)

    return summary

# Example usage
bowling_card = calculate_bowling_card(df)

In [191]:
bowling_card.sample()

,match_id,bowler,bowler_id,bowling_team,batting_team,innings,overs,balls,wickets,runs,lbw_or_bowled,dot_balls,economy,maidens,venue,date,league
49593,1226869,MP Stoinis,325012.0,Melbourne Stars,Melbourne Renegades,2,3.0,18,1,22.0,1,5,7.333333,0,Docklands Stadium,2021-01-20,BBL


In [195]:
bowling_card[bowling_card['match_id'] == 287877]

,match_id,bowler,bowler_id,bowling_team,batting_team,innings,overs,balls,wickets,runs,lbw_or_bowled,dot_balls,economy,maidens,venue,date,league
432,287877,Mohammad Asif,41411.0,Pakistan,New Zealand,1,4.0,24,0,34.0,0,8,8.500000,0,Newlands,2007-09-22,T20IS
437,287877,Sohail Tanvir,43265.0,Pakistan,New Zealand,1,4.0,27,0,30.0,0,9,7.500000,0,Newlands,2007-09-22,T20IS
436,287877,Shahid Afridi,42639.0,Pakistan,New Zealand,1,4.0,26,1,24.0,1,8,6.000000,0,Newlands,2007-09-22,T20IS
428,287877,Fawad Alam,40250.0,Pakistan,New Zealand,1,3.0,18,2,29.0,0,4,9.666667,0,Newlands,2007-09-22,T20IS
438,287877,Umar Gul,43524.0,Pakistan,New Zealand,1,4.0,24,3,15.0,0,14,3.750000,0,Newlands,2007-09-22,T20IS
433,287877,Mohammad Hafeez,41434.0,Pakistan,New Zealand,1,1.0,6,0,10.0,0,2,10.000000,0,Newlands,2007-09-22,T20IS
435,287877,SE Bond,36326.0,New Zealand,Pakistan,2,4.0,25,1,33.0,0,12,8.250000,0,Newlands,2007-09-22,T20IS
431,287877,MR Gillespie,37111.0,New Zealand,Pakistan,2,3.0,19,0,30.0,0,6,10.000000,0,Newlands,2007-09-22,T20IS
427,287877,DL Vettori,38710.0,New Zealand,Pakistan,2,4.0,24,1,22.0,0,14,5.500000,0,Newlands,2007-09-22,T20IS
434,287877,SB Styris,38407.0,New Zealand,Pakistan,2,3.0,18,1,14.0,1,10,4.666667,0,Newlands,2007-09-22,T20IS


In [196]:
bowling_card.rename(columns={'batting_team':'bat_team', 'bowling_team':'bowl_team', 'innings':'inns', 'economy':'eco', 'dot_balls':'dots'}, inplace=True)

In [197]:
bowling_card.columns

Index(['match_id', 'bowler', 'bowler_id', 'bowl_team', 'bat_team', 'inns',
       'overs', 'balls', 'wickets', 'runs', 'lbw_or_bowled', 'dots', 'eco',
       'maidens', 'venue', 'date', 'league'],
      dtype='object')

In [198]:
batting_card.to_csv(r"D:\all_league\all_league_batting_card.csv", index=False)
bowling_card.to_csv(r"D:\all_league\all_league_bowling_card.csv", index=False)